# PD-EEG × S-CEReBrO 검증 실행 (2026-11-12)

**런타임 → 런타임 유형 변경 → GPU (T4)** 로 설정한 뒤 위에서부터 차례로 실행하세요.

| 단계 | 내용 | 통과 기준 |
|---|---|---|
| 0 | 준비: Drive 연결, 코드 복사, 설치, 경로 | `run.py configs medley` 가 보임 |
| 1 | PD-EEG 원본 전체 복사 (백그라운드) | 폴더 144개, 복사 프로세스 종료 |
| 2 | S-CEReBrO 입력이 BioFoundation 원본과 같은지 확인 | 좌표·정규화 차이 0, 임베딩 차이 ≤ 1e-5 |
| 3 | 전체 추출 → `cv` | 116명 추출, CBraMod 와 같은 113명·같은 split |
| 4 | CBraMod vs S-CEReBrO 비교 표 | — |

전제: Drive 에 `medley_pd/medley_pd/` (코드), `medley_pd/features/pd_eeg/cbramod/`, `medley_pd/splits/pd_eeg__moca.csv`, `medley_pd/results/pd_eeg/moca/cbramod/` 가 이미 있음 (어제 실행 결과).

## 0. 준비

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess, sys
WORK = '/content/drive/MyDrive/medley_pd'                  # 피처·결과·split (Drive)
CODE = '/content/medley_pd'                                # 코드 (로컬 복사본)
DATA = '/content/data'                                     # EEG 원본 (로컬, 빠른 읽기)
OLD  = '/content/drive/MyDrive/pd_eeg_project/ds007526'    # PD-EEG 원본 (Drive)

%cd /content
!rm -rf {CODE}
!cp -r {WORK}/medley_pd {CODE}
%cd {CODE}
!ls
os.makedirs(f'{DATA}/ds007526', exist_ok=True)
!cp {OLD}/participants.* {DATA}/ds007526/

In [ ]:
!pip install -q mne einops huggingface_hub pyyaml timm safetensors lmdb
import torch; print('GPU:', torch.cuda.is_available())

In [ ]:
# 공통 경로 옵션 + S-CEReBrO 코드(BioFoundation PR #18, commit 361c017)와 가중치
S = (f'--set paths.data_root={DATA} --set paths.feature_root={WORK}/features '
     f'--set paths.result_root={WORK}/results --set paths.split_root={WORK}/splits')

BF = f'{WORK}/BioFoundation'
if not os.path.exists(f'{BF}/models/s_cerebro.py'):
    !git clone -q https://github.com/pulp-bio/BioFoundation "$BF"
    !cd "$BF" && git fetch -q origin 361c017ce0018141391d068e85fd6352fa32a9ef && git checkout -q FETCH_HEAD
!cd "$BF" && git log -1 --format='BioFoundation commit: %H'
from huggingface_hub import hf_hub_download
SCW = hf_hub_download('PulpBio/S-CEReBrO', 'S_CEReBrO.safetensors', local_dir=f'{WORK}/S-CEReBrO')
SC = f'--set encoder.repo={BF} --set encoder.weights={SCW}'
print(SC)

## 1. PD-EEG 원본 전체 복사 (백그라운드)
Drive → 로컬 복사는 15~30분 걸립니다. 실행하면 바로 다음 셀(2단계)로 넘어가도 됩니다.

In [ ]:
# 한 번만 실행하세요 (두 번 실행하면 복사 프로세스가 두 개 돌아 File exists 메시지가 납니다)
proc = subprocess.Popen(f'cp -rn {OLD}/sub-* {DATA}/ds007526/ > /content/copy.log 2>&1', shell=True)
print('복사 시작')

In [ ]:
# 진행 확인 (여러 번 실행해도 됨)
import glob
n = len(glob.glob(f'{DATA}/ds007526/sub-*'))
print('복사 중' if proc.poll() is None else f'복사 종료 (exit {proc.returncode})', f'| 피험자 폴더 {n}/144')
!du -sh {DATA}/ds007526
!tail -3 /content/copy.log

## 2. S-CEReBrO 입력이 BioFoundation 원본과 같은지 확인
같은 EEG 구간(sub-030)을 원본 함수와 우리 어댑터에 넣어 비교합니다. sub-030 파일이 아직 복사되지 않았으면 첫 줄이 먼저 복사합니다.

- (1) 전극 좌표: `make_datasets.common.referential_coordinates` vs 우리 계산 → **0**
- (2) 정규화: `LMDBDataset._min_max_normalize` vs 우리 `minmax` → **0**
- (3) 임베딩: 원본 방식으로 모델 직접 호출 vs 우리 `embed()` → **≤ 1e-5**
- (4) 참고: 필터·리샘플링 순서 차이의 영향 (아래 설명)

In [ ]:
if not os.path.exists(f'{DATA}/ds007526/sub-030/eeg/sub-030_task-rest_eeg.set'):
    !cp -r {OLD}/sub-030 {DATA}/ds007526/
import numpy as np, torch
sys.path.insert(0, BF)
import importlib.util
def _load(name, path):     # Colab 의 Hugging Face `datasets` 패키지와 이름이 겹쳐 파일 경로로 직접 불러옴
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec); sys.modules[name] = mod; spec.loader.exec_module(mod)
    return mod
bf_coords = _load('bf_common', f'{BF}/make_datasets/common.py').referential_coordinates
LMDBDataset = _load('bf_lmdb_dataset', f'{BF}/datasets/lmdb_dataset.py').LMDBDataset
from medley.config import load_config
from medley.eeg import preprocess
from medley.encoders.scerebro import SCEReBrO, referential_coordinates as my_coords, minmax

ov = [x for x in (S + ' ' + SC).replace('--set ', '').split()]
cfg = load_config('pd_eeg', 'scerebro', overrides=ov)
EEGF = f'{DATA}/ds007526/sub-030/eeg/sub-030_task-rest_eeg.set'
segs, flag, info = preprocess(EEGF, cfg)
print('구간', segs.shape, info)

ch = [c.upper() for c in cfg['encoder']['channels']]
a, b = my_coords(BF, cfg['encoder']['channels'], 'CZ'), bf_coords(ch, 'CZ')
print('(1) 좌표 최대 차이:', float(np.abs(a - b).max()))

x = segs[:4]
mine = minmax(x)
theirs = torch.stack([LMDBDataset._min_max_normalize(None, torch.from_numpy(s.copy())) for s in x]).numpy()
print('(2) 정규화 최대 차이:', float(np.abs(mine - theirs).max()))

enc = SCEReBrO(cfg['encoder'])
xb = torch.from_numpy(theirs.reshape(4, 19, -1, 200)).float().to(enc.device)
pos = torch.from_numpy(b).float().to(enc.device).unsqueeze(0).expand(4, -1, -1, -1)
with torch.no_grad():
    tok = enc.model(xb, channel_positions=pos, directly_input_tokens=False, attn_mask=None)
ref = tok.reshape(4, 19, -1, tok.shape[-1]).mean(2).reshape(4, -1).cpu().numpy()
emb = enc.embed(x)
print('(3) 임베딩 최대 차이:', float(np.abs(emb - ref).max()), '| 임베딩 크기 평균', float(np.abs(ref).mean()),
      '| 모양', emb.shape, '| NaN', bool(np.isnan(emb).any()))

**(4) 참고 — 필터와 리샘플링 순서.** BioFoundation 의 `make_mumtaz.py` 는 *리샘플링 → 필터 → 노치* 순서이고, 우리 공통 전처리(CBraMod 프로토콜)는 *필터 → 노치 → 리샘플링* 순서입니다. 두 인코더를 같은 전처리로 비교하려고 공통 순서를 유지했고, 아래 셀은 그 차이가 임베딩에 얼마나 영향을 주는지 숫자로 남깁니다. 상관이 0.99 이상이면 무시할 수준입니다.

In [ ]:
from medley.eeg import load_raw
raw, _ = load_raw(EEGF, cfg['encoder']['channels'], cfg['dataset'].get('ref_aliases', []))
raw.resample(200, verbose='ERROR'); raw.filter(0.3, 75, verbose='ERROR'); raw.notch_filter(50, verbose='ERROR')
xo = raw.get_data() * 1e6
n = xo.shape[1] // 1000
segs_bf = xo[:, :n * 1000].reshape(19, n, 1000).transpose(1, 0, 2).astype(np.float32)
m = min(len(segs_bf), len(segs))
e1, e2 = enc.embed(segs[:m]), enc.embed(segs_bf[:m])
r = np.corrcoef(e1.mean(0), e2.mean(0))[0, 1]
print(f'(4) 구간 {m}개 | 피험자 평균 임베딩 상관 {r:.5f} | 상대 차이 {np.abs(e1 - e2).mean() / np.abs(e2).mean():.4f}')

## 3. 전체 추출 → `cv`
1단계 복사가 끝난 뒤(폴더 144/144, 복사 종료) 실행하세요.

In [ ]:
import glob
assert len(glob.glob(f'{DATA}/ds007526/sub-*')) == 144, '아직 복사 중입니다. 1단계 진행 확인 셀을 다시 실행하세요.'
!ps aux | grep "cp -rn" | grep -v grep | wc -l     # 0 이어야 복사 완료
!python run.py extract --dataset pd_eeg --encoder scerebro $S $SC 2>&1 | grep -v FutureWarning | grep -v make_standard_montage | tail -8

In [ ]:
# 추출 결과 점검: 116명 전원, 구간 수, NaN, 보간 채널
import pandas as pd
log = pd.read_csv(f'{WORK}/features/pd_eeg/scerebro/extract_log.csv')
print(log.status.value_counts().to_string())
bad = []
for sid in log.subject:
    z = np.load(f'{WORK}/features/pd_eeg/scerebro/{sid}.npz')['feat']
    if z.shape[1] != 3420 or np.isnan(z).any(): bad.append(sid)
print('구간 수 분포:', log.n_seg.value_counts().to_dict())
print('보간 채널:', log.interp.fillna('').value_counts().to_dict() if 'interp' in log else '-')
print('모양·NaN 이상:', bad or '없음')

In [ ]:
!python run.py cv --dataset pd_eeg --encoder scerebro --task moca $S 2>&1 | grep -v Warning

## 4. CBraMod vs S-CEReBrO 비교
`run.py compare`: 같은 113명·같은 seed 로 짝지어 비교합니다. Δ = S-CEReBrO − CBraMod (MAE 는 음수면 S-CEReBrO 가 나음). EEG 를 쓰지 않는 모델의 예측이 두 인코더에서 다르면 멈춥니다.

In [ ]:
import pandas as pd
!python run.py compare --dataset pd_eeg --task moca --encoders cbramod scerebro $S

R = f'{WORK}/results/pd_eeg/moca'
for e in ['cbramod', 'scerebro']:
    s = pd.read_csv(f'{R}/{e}/summary.csv')
    print(f'\n== {e}')
    print(s[['model', 'avg_MAE', 'MAE_lo', 'MAE_hi', 'avg_Spearman', 'avg_R2']].round(3).to_string(index=False))

## 5. (선택) S-CEReBrO 누출 점검
새 인코더로 처음 돌린 결과이므로 MoCA 를 섞어 한 번 더 확인합니다. 모든 모델 MAE ≈ mean, R² ≈ 0 이면 통과.

In [ ]:
t = pd.read_csv(f'{DATA}/ds007526/participants.tsv', sep='\t')
mk = t.group.eq('PD') & t.moca.notna()
t.loc[mk, 'moca'] = np.random.default_rng(0).permutation(t.loc[mk, 'moca'].values)
os.makedirs(f'{DATA}/ds007526_shuf', exist_ok=True)
t.to_csv(f'{DATA}/ds007526_shuf/participants.tsv', sep='\t', index=False)
!python run.py cv --dataset pd_eeg --encoder scerebro --task moca $S --set dataset.dir=ds007526_shuf --set paths.split_root=/content/tmp_splits --set paths.result_root=/content/tmp_results --set "evaluation.seeds=[0,1,2]" 2>&1 | sed -n '/모델별 성능/,/paired/p'